In [20]:
import pandas as pd
from bs4 import BeautifulSoup
import requests

def get_page(isbn):
    url = r"https://www.yes24.com/product/search?domain=ALL&query={}"
    r = requests.get(url.format(isbn))
    soup = BeautifulSoup(r.text, 'html.parser')
    detail_page = soup.find('a', {'class':'gd_name'})
    if detail_page == None:
        return "상세페이지 없음!" 
    
    url = r"https://www.yes24.com" + detail_page['href']
    r = requests.get(url)
    soup = BeautifulSoup(r.text, 'html.parser')
    div = soup.find('div', {'id':'infoset_specific'})
    tr_list = div.find_all('tr')
    for tr in tr_list:
        if tr.find('th').get_text() == "쪽수, 무게, 크기":
            result = tr.find('td').get_text().split()[0]
            return result
    return ""
    
books = pd.read_json(r"C:\data\20s_book_2025.json")
top10_books = books.head(10)

def apply_isbn(row):
    isbn = row['isbn13']
    return get_page(isbn)

top10_books_page = top10_books.apply(lambda row: get_page(row['isbn13']), axis = 1)
top10_books_page.name = 'page'
top10_books_with_page = pd.merge(top10_books, top10_books_page, right_index = True, left_index = True)

page_list = [int(p.replace('쪽', '')) for p in top10_books_with_page['page']]
page_tot = sum(page_list)
page_avg = page_tot/len(page_list)
print(page_avg)

320.4
